# MCP Server & Client Implementation Using Fastmcp 

---
## Architecture

LLM Host (MCP Client)  <—— JSON-RPC ——>  fastmcp Server  <——> CSV Dataset


We'll:
1. Launch a server exposing two tools (`summarize`, `query`).
2. Use an MCP Client to discover and call those tools.
3. Simulate a host (agent) deciding which tool to call.
---
## Start the MCP Server

In a separate terminal, run:
```bash
python mcp_server_fastmcp.py
```
This starts an MCP-compliant server exposing `summarize` and `query` tools.

---


### 1. Install dependencies

In [1]:

%pip install fastmcp pandas openai 
# uvicorn --upgrade --no-cache-dir

Note: you may need to restart the kernel to use updated packages.


## Connect via MCP Client

Once the server is running, connect using the MCP client to list tools and invoke them.

In [2]:
from fastmcp import Client
from fastmcp.client import PythonStdioTransport
import asyncio

async def main():
    # Create a client using the PythonStdioTransport
    # Create a loop to run the client

    transport = PythonStdioTransport("mcp_server_fastmcp.py")
    
    # client = Client(transport)

    # client = Client(transport)
    async with Client(transport) as client:
        tools = await client.list_tools()
        print("Tools:", [t.name for t in tools])

await main()


Tools: ['ping', 'summarize', 'query']


## Simulate a Simple Host Agent
Let's simulate a rule-based 'AI agent' that decides whether to use `summarize` or `query` based on user text.

In [3]:
from openai import OpenAI
import os

from dotenv import load_dotenv

load_dotenv(override=True, dotenv_path="../.env.local")
my_api_key = os.getenv("OPENAI_API_KEY")
openai_client = OpenAI(api_key=my_api_key)

In [6]:

def decide_tool(text: str):
    # Call LLM and decide which tool to use based on the user input
    # options are "ping", "summarize" and "query"
    # For example, if the user input contains "summarize" or "overview", 
    # we can call the summarize tool.
    # If the user input contains "west", we can call the query tool 
    # with expr = "region == 'West'".
    # Otherwise, we can call the summarize tool by default.
   
   
    response = openai_client.chat.completions.create(
            model="gpt-4.1-nano", #gpt-5-nano, gpt-5-mini, gpt-5, gpt-5-pro
            messages=[
                
                {"role": "system", 
                "content": '''You are a helpful assistant. 
                            decide which tool to use based on the user input
                            options are "ping", "summarize" and "query"
                            For example, if the user input contains "summarize" or "overview", 
                            we can call the summarize tool.
                            If the user input contains "west", we can call the query tool 
                            with expr = "region == 'West'".
                            Otherwise, we can call the summarize tool by default.
                            return the tool name and parameters in the following format:
                            tool_name
                            '''},
                {"role": "user", "content": text}
            ]
        )
    
    return_value = response.choices[0].message.content.strip()
    if "summarize" in return_value:
        return "summarize", {}
    if "query" in return_value and "west" in text.lower():
        return "query", {"expr": "region == 'West'"}
    return "summarize", {}

    
    return return_value, {} # Assuming the tool name is in the first line and parameters are empty for simplicity    
    

async def run_agent(user_input, client):
    tool, params = decide_tool(user_input) #"summarize", {}
    #tool = "summarize"
    #params = {}
    print(f"Agent decided to use '{tool}'")

    # API for fastmcp 2.12.5
    result = await client.call_tool(tool, params)
    # result = await client.call_tool("query", params)

    print("Result:", result, "\n")



In [8]:
# create a connection to your MCP server
transport = PythonStdioTransport("mcp_server_fastmcp.py")

async with Client(transport) as client:
    await run_agent("Give me a summary of the dataset", client)



RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
# create a connection to your MCP server
transport = PythonStdioTransport("mcp_server_fastmcp.py")

async with Client(transport) as client:
    await run_agent("Give me a summary of the dataset", client)
    await run_agent("Show West region sales > 1500", client)

---
##  Architecture Recap
```
+--------------------+      JSON-RPC (MCP)      +--------------------+
|  Host / AI Agent   |  <-------------------->  | fastmcp MCP Server |
| (LLM or Router)    |                        | (summarize, query)  |
+--------------------+                         +--------------------+
```
---